# Python for Neuroscience (2026)


# 06. Two-photon calcium imaging: cell segmentation and signal extraction

<!-- <img src="Images/2P_ca_invivo.png" width="800"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/2P_ca_invivo.png" width="800">

**Figure**. Representative steps of an in vivo two-photon calcium imaging experiment in mice.

After recording the fluorescence signal of the calcium sensor using two-photon calcium imaging, we need to extract the signal from specific neuronal compartments, generally the soma. Cellular segmentation was traditionally done manually or semi-automatically, until the recent development of computer vision algorithms and machine learning methods that allow more automated approaches.

<!-- <img src="Images/06_2P_ca_segmentation.png" width="400"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/06_2P_ca_segmentation.png" width="400">

**GOAL**: In this notebook, we will first see how to use ROIs previously defined with the software [ImageJ](https://imagej.net/ij/docs/index.html) to extract calcium traces from neurons. We will then apply an automatic segmentation approach using [cellpose](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1), a deep learning–based algorithm for cellular segmentation. With both approaches, we will extract the fluorescence signal and calculate ΔF/F traces.

**Further reading**
- Ali and Kwan, 2019. Interpreting in vivo calcium signals from neuronal cell bodies, axons, and dendrites: a review. [Link](https://pmc.ncbi.nlm.nih.gov/articles/PMC6664352/).
- Chen et al., 2013. Ultrasensitive fluorescent proteins for imaging neuronal activity. [Link](https://www.nature.com/articles/nature12354).
- Grienberger et al., 2022. Two-photon calcium imaging of neuronal activity. [Link](https://pmc.ncbi.nlm.nih.gov/articles/PMC10732251/). *Accessible and comprehensive introduction to two-photon calcium imaging.*
- Grienberger and Konnerth, 2012. Imaging Calcium in Neurons. [Link](https://doi.org/10.1016/j.neuron.2012.02.011).
- Scientifica. Multiphoton microscope light path: [part 1](https://www.scientifica.uk.com/learning-zone/introduction-to-a-deeper-look-part-1-the-path-of-light-up-to-your-multiphoton-microscope) and [part 2](https://www.scientifica.uk.com/learning-zone/introduction-to-a-deeper-look-part-2-the-path-of-light-through-your-multiphoton-microscope). *Great introduction to the components of a typical optical path in multiphoton microscopy.*

# Import the libraries

**Documentation**:
- [OpenCV](https://docs.opencv.org/4.x/da/df6/tutorial_py_table_of_contents_setup.html)

Download [ImageJ](https://imagej.net/ij/download.html) to open the TIFF stacks.

In [ ]:
!pip install -U "roifile[all]"

In [ ]:
import numpy as np
import re
import os
import sys
import pandas as pd

import matplotlib.pyplot as plt
# %matplotlib widget  # Interactive plots
plt.rcParams['svg.fonttype'] = 'none' # To generate editable text in saved *.svg plots
plt.close('all')

# Import ROIs from ImageJ
from roifile import ImagejRoi

import cv2 as cv

# Skimage
from skimage import io
from skimage.measure import find_contours
from scipy import signal

# Create the paths

**Data management: Resources**
- [FAIR Guiding Principles for scientific data management and stewardship](https://www.go-fair.org/fair-principles/)
- [NIN guidelines for data storage](https://nin.nl/wp-content/uploads/sites/2/2024/10/data-storage-protocol-NIN_20211210.pdf).
- [NIH data management](https://grants.nih.gov/policy-and-compliance/policy-topics/sharing-policies/dms/data-management).
- [Cambridge research data management](https://www.data.cam.ac.uk/about).

In [ ]:
# Create a folder name for the patch-clamp notebooks
notebook_name = '2P_calcium'

# Current working directory (default is '/content' in Colab)
data_path = os.getcwd()

# Change the folder names accordingly
paths = {'data':  f'{data_path}/Example Data',
         'processed_data': f'{data_path}/Processed_data/{notebook_name}',
         'analysis': f'{data_path}/Analysis/{notebook_name}'}

# Make folders if they do not exist yet
for path in paths.values():
    os.makedirs(path, exist_ok=True)

# Download the data

In [ ]:
if 'google.colab' in str(get_ipython()):

    !wget -O "Example Data/v1_ca_02_mc.tif" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/v1_ca_02_mc.tif"

    !wget -O "Example Data/v1_ca_02_mc_rois.zip" \
    "https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Example%20data/v1_ca_02_mc_rois.zip"
else:
    print("Download data from 'Example Data' folder.")

# Load the data

<!-- <img src="Images/06_v1_ca_02_mc.png" width="300"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/06_v1_ca_02_mc.png" width="300">

**Example data**: Segment of an in vivo two-photon calcium recording of neurons from the mouse primary visual cortex. The recording was motion-corrected using the package [CaImAn](https://caiman.readthedocs.io/en/latest/CaImAn_Tips.html), and downsampled to a frequency of 7-10 Hz to reduce file size.
* `2P_GCaMP8m`. Recorded with GCaMP8m calcium sensor ([Zhang et al., 2023](https://www.nature.com/articles/s41586-023-05828-9)).

In [ ]:
# Filename
experiment = 'v1_ca'
recording = '_02_mc'

# Call this variable stack if you do not need to overwrite it
stack = io.imread(f"{paths['data']}/{experiment}{recording}.tif")

# Stack shape
print("time, x, y dimensions:", stack.shape)

# Load manual cell segmentation

<!-- <img src="Images/06_2P_ca_rois.png" width="300"> -->

<img src="https://raw.githubusercontent.com/dav1dcg/neuro-notebook-templates/main/Python4Neuro_2026/Images/06_2P_ca_rois.png" width="300">

We use the package [roifile](https://github.com/cgohlke/roifile) to load ROIs previously made with the imaging software [ImageJ](https://imagej.net/ij/).

In [ ]:
# Load ROIs
rois_path = f"{paths['data']}/{experiment}{recording}_rois.zip"

ca_rois = ImagejRoi.fromfile(rois_path)

print(ca_rois)

## Create a mask

- [OpenCV draw contours](https://docs.opencv.org/3.4/d4/d73/tutorial_py_contours_begin.html). Function to joing all the continuous points along the boundary.

In [ ]:
def roi_mask(ROI, image_height, image_width):
    '''
    Gets the outline coordinates of a ROI and makes mask image.

    Parameters:
        ROI: Can be either of two objects:
                a) ROI object generated by the package roifile
                b) An outline numpy array as generated by aligning cortical
                    maps to a specific experiment/recording
        image_height, image_width: image dimensions as number of pixels

    Output:
        Mask image, 8 bit with the dimensions of the original image
        where masked pixels are 0 and used pixels 255.
    '''
    mask = np.zeros((image_height, image_width))
    if type(ROI) != np.ndarray:
        contour = ROI.coordinates()  # Method of the package roifile to generate the coordinates of a ROI's outline (contour)
    else:
        contour = ROI  # In the case of an outline ROI is a contour numpy array

    # Takes a contour created by the roifile method coordinates() and selects all pixels within the contour
    mask = cv.drawContours(mask, [contour.astype(int)], 0, (255,255,255), -1)

    return np.uint8(mask)

## Plot the masks

In [ ]:
# Max projection
avg_image = stack.mean(axis=0)

# Store all ROI masks
roi_masks = []

image_height = stack.shape[1]
image_width = stack.shape[2]

for ROI in ca_rois:
    # Apply function to convert ROIs to masks
    mask = roi_mask(ROI, image_height, image_width)
    # Make the mask binary: 1 inside the ROI, 0 outside
    mask_binary = (mask > 1).astype(np.uint8)
    # Append the masks
    roi_masks.append(mask_binary)

# Plot
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')  # Background

# Overlay masks without hiding background
for mask in roi_masks:
    masked = np.ma.masked_where(mask == 0, mask)  # hide zeros
    ax.imshow(masked, cmap='Greens', alpha=0.5)   # overlay only ROI

ax.axis('off')
plt.show()

## Find contours
- https://scikit-image.org/docs/0.23.x/auto_examples/edges/plot_contours.html

**Tips**:
- `contours = find_contours(mask, 0.5)`
- contour[:, 0]. x-coordinates of the contour points.
- contour[:, 1]. y-coordinates of the contour points.

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Loop through ROIs and overlay masks as contours
image_height = stack.shape[1]
image_width = stack.shape[2]

for i, ROI in enumerate(ca_rois):
    # Create mask
    mask = roi_mask(ROI, image_height, image_width)

    # Find contour of the mask
    # contours =  COMPLETE THE CODE
    contours = find_contours(mask, 0.5)

    # Plot each contour
    # for contour in...
      # COMPLETE THE CODE: Plot the contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='white', linewidth=1)

ax.set_title('ROI masks')
ax.axis('off')  # Optional
plt.show()

<details>
<summary><strong>Solution</strong></summary>

```python

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Loop through ROIs and overlay masks as contours
image_height = stack.shape[1]
image_width = stack.shape[2]

for i, ROI in enumerate(ca_rois):
    # Create mask
    mask = roi_mask(ROI, image_height, image_width)
    
    # Find contour of the mask
    contours = find_contours(mask, 0.5)
    
    # Plot each contour
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

ax.set_title('ROI masks')
ax.axis('off')
plt.show()

```

## Calculate "centroid" of the ROI

We calculate the mean of the ROI pixel coordinates, which approximates the center.

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute "centroid" of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        # Compute the average x and y: center of the ROI
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Print the index and coordinates
        print(f"ROI {i+1}: x = {x_mean:.1f}, y = {y_mean:.1f}")

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

## Q: Annotate the ROIs

**Tips**:
- Create list at top `roi_labels` to store the labels
- Append the labels
- Use `ax.text`: https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
# INSERT: List for roi_labels

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Print the index and coordinates
        print(f"ROI {i+1}: x = {x_mean:.1f}, y = {y_mean:.1f}")

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='right', va='top')

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

<details>
<summary><strong>Solution</strong></summary>

```python

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)
    
    # Find contours for visualization
    contours = find_contours(mask, 0.5)
    
    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)
    
    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Print the index and coordinates
        print(f"ROI {i+1}: x = {x_mean:.1f}, y = {y_mean:.1f}")

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))
```

## Extract trace from ROIs

In [ ]:
def trace_from_mask(stack, mask):
    '''
    Makes a trace of the average pixel values defined by a mask image
    Parameters:
        stack: image stack
        mask: image where background pixels are 0 and the pixels of interest > 0.
    Output:
        Trace as a 1D numpy array
    '''

    stack_copy = stack.copy().astype(np.float32)  # Float needed for NaN
    stack_copy[:, mask == 0] = np.nan             # Background pixels to NaN
    trace = np.nanmean(stack_copy, axis=(1, 2))  # Mean over ROI pixels only

    return trace

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # GET TRACES FROM ROIS
        f = trace_from_mask(stack, mask)

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

## Q: Plot the traces from ROIs

**Tips**:
- Store the traces in dictionaries, e.g.: `roi_traces = []`
- Append the traces

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # GET TRACES FROM ROIS
        trace = trace_from_mask(stack, mask)

        # APPEND THE TRACES
        roi_traces.append(trace)

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# Plot the traces
n_rois = len(roi_traces) # COMPLETE THE CODE (Set the length of the plots to the number of traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# Plot each trace (LOOP), add roi_traces, and roi_labels
for i in range(n_rois):
  # Plot each trace
    ax[i].plot(roi_traces[i])  # ax[i]... roi_traces
    ax[i].set_ylabel(roi_labels[i])  # set_ylabel..
    ax[i].set_xlim(0, len(roi_traces[i]))  # set_xlim 0 to length of roi traces

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

<details>
<summary><strong>Solution</strong></summary>

```python

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)
    
    # Find contours for visualization
    contours = find_contours(mask, 0.5)
    
    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)
    
    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # GET TRACES FROM ROIS
        trace = trace_from_mask(stack, mask)

        # APPEND THE TRACES
        roi_traces.append(trace)  

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()
```

## Q: Calculate deltaF/F traces: plot and save as dataframe

Read about [delta/F normalization](https://www.scientifica.uk.com/learning-zone/how-to-compute-%CE%B4f-f-from-calcium-imaging-data) (Peter Rupprecht).

**Tips**:
- trace_deltaF = (trace - f0) / f0 * 100, but avoid dividing by 0!
- Create the dataframe based on the number of rois
- [df.to_csv](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) to save traces as df.
- roi_traces are a list that you need to conver to np.array: `np.array(roi_traces)`. Check shape if you need to [transpose](https://numpy.org/doc/stable/reference/generated/numpy.transpose.html) it

In [ ]:

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # Get traces from ROIs
        trace = trace_from_mask(stack, mask)

        # CALCULATE DELTA/F
        f0 = np.median(trace)  # baseline
        if f0 != 0:
            trace_deltaf = (trace - f0) / f0 * 100
        else:
            trace_deltaf = np.zeros_like(trace)

        # Append the traces
        roi_traces.append(trace_deltaf)

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# SAVE THE TRACES AS DATAFRAME

ca_deltaF_traces = pd.DataFrame(np.array(roi_traces).T, columns=roi_labels)

# OPTIONAL, add column with frames
frames = np.arange(1, stack.shape[0] + 1)
ca_deltaF_traces.insert(0, "Frame", frames) # Insert as the first column in the DataFrame
ca_deltaF_traces.columns = ["Frame"] + roi_labels  # rename columns

# Save CSV
ca_deltaF_traces.to_csv(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.csv", index=False)

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

ca_deltaF_traces

<details>
<summary><strong>Solution</strong></summary>

```python

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)
    
    # Find contours for visualization
    contours = find_contours(mask, 0.5)
    
    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)
    
    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # Get traces from ROIs
        trace = trace_from_mask(stack, mask)

        # CALCULATE DELTA/F
        f0 = np.median(trace)  # baseline
        if f0 != 0:
            trace_deltaf = (trace - f0) / f0 * 100
        else:
            trace_deltaf = np.zeros_like(trace)

        # Append the traces
        roi_traces.append(trace_deltaf)  

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# SAVE THE TRACES AS DATAFRAME

ca_deltaF_traces = pd.DataFrame(np.array(roi_traces).T, columns=roi_labels)

# OPTIONAL, add column with frames
frames = np.arange(1, stack.shape[0] + 1)
ca_deltaF_traces.insert(0, "Frame", frames) # Insert as the first column in the DataFrame
ca_deltaF_traces.columns = ["Frame"] + roi_labels  # rename columns

# Save CSV
ca_deltaF_traces.to_csv(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.csv", index=False)

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

ca_deltaF_traces
```

## Process the traces: baseline correction

Stack can be also prefiltered, code for removing artifacts, etc.

Detrend to correct traces that show a linear trend in their baseline, e.g. as a consequence of bleaching.

**Tips**:
- You can use scipy.signal detrend: https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.detrend.html



In [ ]:
def timeseries_detrend(trace_data, window=None):
    """
    Detrend a time-series trace using various methods.

    Args:
        trace_data (numpy array or pd.Series): The data trace to be detrended.
        window (int, optional): The window size for rolling linear correction or other methods (optional).

    Returns:
        numpy array: Detrended trace.
    """

    # Convert pandas Series to numpy array if needed
    if isinstance(trace_data, pd.Series):
        trace_data = trace_data.to_numpy()

    # Detrend trace in segments if window is given
    if window is not None:
        detrended_segments = [signal.detrend(trace_data[i:i + window], type='linear')
                              for i in range(0, len(trace_data), window)]  # Loop over trace in steps of 'window'

        # Combine all the detrended segments back into one array
        return np.concatenate(detrended_segments)
    else:
        return signal.detrend(trace_data, type='linear')

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # Get traces from ROIs
        trace = trace_from_mask(stack, mask)

        # Calculate deltaF/F
        f0 = np.median(trace)  # baseline
        if f0 != 0:
            trace_deltaf = (trace - f0) / f0 * 100
        else:
            trace_deltaf = np.zeros_like(trace, window=50)

        # BASELINE CORRECTION
        trace_deltaf = timeseries_detrend(trace_deltaf)

        # Append the traces
        roi_traces.append(trace_deltaf)

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# SAVE THE TRACES AS DATAFRAME
ca_deltaF_traces = pd.DataFrame(np.array(roi_traces).T, columns=roi_labels)

# OPTIONAL, add column with frames
frames = np.arange(1, stack.shape[0] + 1)
ca_deltaF_traces.insert(0, "Frame", frames) # Insert as the first column in the DataFrame
ca_deltaF_traces.columns = ["Frame"] + roi_labels  # rename columns

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

ca_deltaF_traces

## Save traces figure and table

In [ ]:
# Save CSV
ca_deltaF_traces.to_csv(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.csv", index=False)
fig.savefig(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.svg")

## Q: Filter the traces before baseline correction

**Note**: Second-order sections (SOS) are a way to represent a digital filter as a series of small, stable 2-pole filters instead of one big filter, so each section is more numerically stable. SOS is useful for high-order filters, long signals, or very small/large numbers.

**Tips**:
- Use [signal.butter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html): N (order of the filter), Wn (point at which the gain drops to 1/sqrt(2) that of the passband), btype (type of filter)
- Calculate Wn first.

In [ ]:
# Compute normalized cutoff frequency
freq_cutoff = 5
fs = 15.3
Wn = freq_cutoff / (fs / 2)
Wn

In [ ]:
# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)

    # Find contours for visualization
    contours = find_contours(mask, 0.5)

    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)

    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # Get traces from ROIs
        trace = trace_from_mask(stack, mask)

        # Calculate deltaF/F
        f0 = np.median(trace)  # baseline
        if f0 != 0:
            trace_deltaf = (trace - f0) / f0 * 100
        else:
            trace_deltaf = np.zeros_like(trace, window=50)

        # FILTER THE TRACES
        # low-pass Butterworth filter in “second-order sections”
        sos = signal.butter(# COMPLETE THE CODE)
        delta_f = signal.sosfiltfilt(# COMPLETE THE CODE)

        # BASELINE CORRECTION
        trace_deltaf = timeseries_detrend(trace_deltaf)

        # Append the traces
        roi_traces.append(trace_deltaf)

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# SAVE THE TRACES AS DATAFRAME
ca_deltaF_traces = pd.DataFrame(np.array(roi_traces).T, columns=roi_labels)

# OPTIONAL, add column with frames
frames = np.arange(1, stack.shape[0] + 1)
ca_deltaF_traces.insert(0, "Frame", frames) # Insert as the first column in the DataFrame
ca_deltaF_traces.columns = ["Frame"] + roi_labels  # rename columns

# Save CSV
ca_deltaF_traces.to_csv(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.csv", index=False)

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

ca_deltaF_traces

<details>
<summary><strong>Solution</strong></summary>

```python

# Compute average image of the stack
avg_image = stack.mean(axis=0)

# Prepare figure
fig, ax = plt.subplots(figsize=(5,5))
ax.imshow(avg_image, cmap='gray')

# Lists to store ROI info
roi_coords = []
roi_labels = []
roi_traces = []

# Image dimensions
image_height = stack.shape[1]
image_width = stack.shape[2]

# Loop through ROIs
for i, ROI in enumerate(ca_rois):
    # Create mask from ROI
    mask = roi_mask(ROI, image_height, image_width)
    
    # Find contours for visualization
    contours = find_contours(mask, 0.5)
    
    # Plot contours
    for contour in contours:
        ax.plot(contour[:, 1], contour[:, 0], color='magenta', linewidth=1)
    
    # Compute centroid of ROI mask
    y_indices, x_indices = np.nonzero(mask)
    if len(x_indices) > 0:
        x_mean = np.mean(x_indices)
        y_mean = np.mean(y_indices)
        roi_coords.append([x_mean, y_mean])

        # Annotate the neuron
        label = str(i + 1)
        roi_labels.append(label)
        ax.text(x_mean, y_mean, label, color='white', fontsize=10,
                ha='center', va='center')

        # Get traces from ROIs
        trace = trace_from_mask(stack, mask)

        # Calculate deltaF/F
        f0 = np.median(trace)  # baseline
        if f0 != 0:
            trace_deltaf = (trace - f0) / f0 * 100
        else:
            trace_deltaf = np.zeros_like(trace, window=50)

        # FILTER THE TRACES
        sos = signal.butter(N=4, Wn=0.65, btype='lowpass', output='sos')
        trace_deltaf = signal.sosfilt(sos, trace_deltaf)

        # BASELINE CORRECTION
        trace_deltaf = timeseries_detrend(trace_deltaf)

        # Append the traces
        roi_traces.append(trace_deltaf)  

ax.set_title('ROI masks')
plt.show()

# Save ROI coordinates
coord_path = f"{paths['analysis']}/{experiment}{recording}_ca_ROIs_coordinates.npy"
np.save(coord_path, np.array(roi_coords))

# SAVE THE TRACES AS DATAFRAME
ca_deltaF_traces = pd.DataFrame(np.array(roi_traces).T, columns=roi_labels)

# OPTIONAL, add column with frames
frames = np.arange(1, stack.shape[0] + 1)
ca_deltaF_traces.insert(0, "Frame", frames) # Insert as the first column in the DataFrame
ca_deltaF_traces.columns = ["Frame"] + roi_labels  # rename columns

# Save CSV
ca_deltaF_traces.to_csv(f"{paths['analysis']}/{experiment}{recording}_ca_deltaF_traces.csv", index=False)

# Plot the traces
n_rois = len(roi_traces)
fig, ax = plt.subplots(n_rois, 1, figsize=(5, n_rois), sharex=True)

# If only one ROI (to avoid errors)
if n_rois == 1:
    ax = [ax]

# Plot each trace
for i in range(n_rois):
    ax[i].plot(roi_traces[i])
    ax[i].set_ylabel(roi_labels[i])
    ax[i].set_xlim(0, len(roi_traces[i]))

ax[-1].set_xlabel("Frames")

plt.tight_layout()
plt.show()

ca_deltaF_traces
```

# Cell segmentation with Cellpose

> "The most important aspect of biological software is that it works well in the hands of biologists." [Pachitariu et al., 2025](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1.full)
>


Here, we use **Cellpose** for cell segmentation of two-photon calcium recordings. Specifically, we will use the latest version developed by Stringer, Pachitariu, and colleagues, **Cellpose-SAM**, which integrates the foundation model **Segment Anything Model (SAM)** ([Kirillov et al., 2023](https://openaccess.thecvf.com/content/ICCV2023/html/Kirillov_Segment_Anything_ICCV_2023_paper.html)) into Cellpose. The original Cellpose used a neural network (U-Net) to predict spatial gradients and then produce masks ([Stringer et al., 2021](https://www.nature.com/articles/s41592-020-01018-x)), whereas Cellpose-SAM adapts SAM's transformer-based architecture to enhance generalization across diverse imaging conditions. For details, see [Pachitariu et al., 2025](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1.full).

**Cellpose** is [well-documented](https://cellpose.readthedocs.io/en/latest/), actively maintained (see the [GitHub repository](https://github.com/MouseLand/cellpose)), and, most importantly, easy to use ([example notebook](https://github.com/MouseLand/cellpose/blob/main/notebooks/run_Cellpose-SAM.ipynb)). Among other outputs, it returns **masks** and regions of interest (ROIs), which can be exported for use in **ImageJ** or Python (ImageJROI).

Post-hoc filtering (e.g., signal-to-noise criteria) can help remove noisy traces and artifacts. In this notebook, I show a simple example of how to apply Cellpose to two short time-series recordings (see below) to calculate and plot the **ΔF/F** traces from the masks identified by Cellpose.



**References**:
- [Cellpose-SAM website](https://huggingface.co/spaces/mouseland/cellpose) To process images online.
- Pachitariu, M., Rariden, M., & Stringer, C. (2025). Cellpose-SAM: superhuman generalization for cellular segmentation. [bioRxiv](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1.full).

## Installation

**IMPORTANT**: You first need to install [cellpose](https://github.com/MouseLand/cellpose) to use this notebook. Please read the up-to-date instructions on the [cellpose GitHub](https://github.com/MouseLand/cellpose/blob/main/README.md/#Installation). In brief, you need:
1. Open an anaconda prompt or command prompt which has `conda` for python 3 in the path
2. Create a new environment: `conda create --name cellpose python=3.10`
3. Activate the new environment: `conda activate cellpose`
4. Install cellpose: `python -m pip install cellpose[gui]`(with the user interface) or `python -m pip install cellpose` (without the user interface).
5. Install the following packages to run this notebook: `python -m pip install jupyterlab matplotlib pandas`
6. If needed: `python -m pip install cellpose --upgrade`

Note: To use the GPU, you need to install the drivers and the CUDA libraries.

**GOOGLE COLAB**: Change runtime type and select GPU

In [ ]:
!pip install cellpose

In [ ]:
# Cellpose functions
from cellpose import models, core, io, utils, plot
from cellpose.utils import masks_to_outlines

io.logger_setup() # run this to get printing of progress
model = models.CellposeModel(gpu=True)

## Segmentation parameters

More information about parameters [here](https://cellpose.readthedocs.io/en/latest/settings.html). Default parameters are already quite good, but tuning them can help reduce "false positives." For further fine-tuning, the authors recommend using your own recordings to train the model, which should improve accuracy (see [how](https://github.com/MouseLand/cellpose/blob/main/docs/train.rst)).

- **flow_threshold**: Maximum allowed error of the flows for each mask. **Default: 0.4**. Increase this threshold if Cellpose is not returning as many masks as expected (or turn off completely with 0.0). Decrease it if Cellpose returns too many ill-shaped masks.
- **cellprob_threshold**: Probability threshold that a detected object is a cell. **Default: 0.0**. Decrease this threshold if Cellpose is not returning enough masks or the masks are too small. Increase it if Cellpose returns too many ROIs, particularly from dim areas.
- **tile_norm_blocksize**: Size of the blocks used for image normalization. **Default: 0.0** (entire image normalized together). Set to 100–200 pixels if brightness is very inhomogeneous across the image.
- **niter**: **Default: None or 0**. Sets the number of iterations proportional to the ROI diameter.


In [ ]:
# Cellpose SAM Parameters
flow_threshold = 0.2  # Maximum allowed error of the flows for each mask
cellprob_threshold = 0.5  # Determines proability that a detected object is a cell
tile_norm_blocksize = 0   # Size of blocks used for normalizing the image
niter = 20  # Number of iterations. Default is proportional to the ROI diameter.

# Create a dictionary
cellpose_params = {
    "flow_threshold": flow_threshold,
    "cellprob_threshold": cellprob_threshold,
    "tile_norm_blocksize": tile_norm_blocksize,
    "niter": niter}

# Save the dictionary to a text file
with open(f"{paths['analysis']}/{experiment}{recording}_cellpose_params.txt" , "w") as f:
    for key, value in cellpose_params.items():
        f.write(f"{key} = {value}\n")

## Loop for two-photon calcium recordings

This is an example loop that you can adapt to your paths and datasets. After running cellpose, it uses the cell masks to extract and plot the ΔF/F for each mask (∼neuron). For a comprehensive guide on ΔF/F calculation in calcium imaging, see this in-depth article by [Peter Rupprecht](https://www.scientifica.uk.com/learning-zone/how-to-compute-%CE%B4f-f-from-calcium-imaging-data).

In [ ]:
%%time

for dirpath, dirnames, files in os.walk(paths['data']):
    for file in files:
        if file.endswith('_02_mc.tif'):
            file_path = f"{dirpath}/{file}"
            filename = os.path.splitext(file)[0]

            # Read image and calculate average image
            stack = io.imread(file_path)
            stack_avg = stack.mean(axis=0)

            # Run Cellpose to segment neurons
            masks, flows, styles = model.eval(
                stack_avg,
                batch_size=32,
                flow_threshold=flow_threshold,
                cellprob_threshold=cellprob_threshold,
                normalize={"tile_norm_blocksize": tile_norm_blocksize},
                niter=niter)

            # Save masks as both image and numpy array
            masks_path = f"{paths['analysis']}/{filename}_masks_cellpose.tif"
            masks_img = io.imsave(masks_path, masks.astype('uint16'))
            np.save(f"{paths['analysis']}/{filename}_masks_cellpose.npy", masks)

            # Save ROIs for ImageJ (open with ImageJ ROI manager)
            io.save_rois(masks, f"{paths['analysis']}/{filename}_cellpose.zip")

            # Segmentation images
            fig = plt.figure(figsize=(12,5))
            plot.show_segmentation(fig, stack_avg, masks, flows[0])
            plt.tight_layout()
            fig.savefig(f"{paths['analysis']}/{filename}_segmentation_cellpose.svg", dpi=300)
            plt.close(fig)

            # Extract calcium traces from cell masks
            cell_masks = np.unique(masks)  # Get all unique region labels in the masks
            cell_masks = cell_masks[cell_masks != 0]  # Ignore 0 values

            traces = []
            for mask in cell_masks:
                cell_mask = (masks == mask)  # Binary mask for the current cell
                mask_pixels = stack[:, cell_mask]  # Get mask pixels over all time frames

                trace = np.nanmean(mask_pixels, axis=1)  # Calculate mean of this cell over time

                # DeltaF/F calculation
                f0 = np.median(trace)  # Or using percentiles
                delta_f = (trace - f0) / f0 * 100 if f0 != 0 else np.zeros_like(trace)
                traces.append(delta_f)

            # Save traces to a dataframe
            df_index = np.arange(stack.shape[0]) + 1  # Frames
            df = pd.DataFrame(np.vstack(traces).T,
                              index=df_index,
                              columns=[f'n_{mask}' for mask in cell_masks])

            csv_path = f"{paths['analysis']}/{filename}_traces_cellpose.csv"
            df.to_csv(csv_path)
            print(f"Dataframe with deltaF_F traces: {csv_path}")

            # Plot deltaF/F traces (adapt as you wish to suit your preferences)
            n_cell_masks = len(cell_masks)  # One trace per row
            fig, axes = plt.subplots(n_cell_masks, 1, figsize=(4, 0.15 * n_cell_masks),
                                     sharex=True, sharey=True)
            if n_cell_masks == 1:
                axes = [axes]  # In case only one neuron is detected

            for ax, trace, cell_mask in zip(axes, traces, cell_masks):
                ax.plot(df_index, trace, color='black', linewidth=0.5)
                ax.set_ylabel(f'n_{cell_mask}', rotation=0, labelpad=10, va='center')
                # Remove spines and tick to make the plot cleaner
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)
                ax.spines['left'].set_visible(False)
                ax.spines['bottom'].set_visible(False)
                ax.set_yticks([])
                ax.set_xticks([])

            plt.subplots_adjust(hspace=0.01, left=0.1, right=0.9, top=0.9, bottom=0.01)
            plt.tight_layout(pad=0.5)
            trace_plot_path = f"{paths['analysis']}/{filename}_traces_cellpose.png"  # or svg
            fig.savefig(trace_plot_path, dpi=300)
            plt.close(fig)
            print(f"Traces plot: {trace_plot_path}")

            del stack  # Remove stack from memory

## Import the cellpose ROIs

Exported ROIs can be opened in ImageJ or in Python with ImageJROI ([read documentation](https://github.com/cgohlke/roifile)). Below is just an example of how to load the file. Alternatively, you can load the NumPy array with the masks, which may be easier for extracting traces if you want to reuse them in another script.


In [ ]:
for dirpath, dirnames, files in os.walk(paths['analysis']):
    for file in files:
        if file.endswith('.zip'):
            file_path = f"{dirpath}/{file}"
            filename = os.path.splitext(file)[0]
            rois_path = f"{paths['analysis']}{filename}.zip"
            # Use ImagejRois to load ROIs from file
            rois = ImagejRoi.fromfile(rois_path)
            for i, roi in enumerate(rois):
                print(roi.coordinates()[0])  # Example to get coordinates from each ROI.

# Open project: Analyze calcium transients indelta/F traces

* Use another **tif** example and extract the traces from ROIs using manual or automatic segmentation.
* Use scipy.find_peaks or other method to find the calcium transients peaks and a few extract properties like amplitude, width, etc.
* Calculate pairwise correlations using [numpy](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html) or [pandas](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html).
* Find cluster of neurons based on correlations using, e.g., [scipy.cluster.hierarchy import dendrogram](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html).